# Monte Carlo Tree Search in Python: Beating a Heuristic Without One

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/monte_carlo_tree_search.ipynb)

Every other reinforcement learning method keeps something: a table, a network, a policy. Monte Carlo tree search keeps nothing. It is handed a position, spends a budget of random playouts, returns a move, and throws the tree away. No training run, no dataset, no evaluation function.

This notebook builds it in about eighty lines and measures it against a hand-written alpha-beta opponent at roughly the same number of nodes, where it wins about seven games in ten while knowing nothing about Connect Four except how to play a legal move.

Everything runs on a CPU in two to three minutes. No GPU, no framework, no dependencies beyond matplotlib.

Companion post: [Monte Carlo Tree Search in Python: Beating a Heuristic Without One](https://sesen.ai/blog/monte-carlo-tree-search-python)

## 1. A game to search

Connect Four's 7x6 grid fits in 49 bits with one sentinel row per column, which makes a move two integer operations and a win check four shifts. That matters only because the search is measured in rollouts per second: a readable 6x7 array board runs the same search roughly forty times slower.

`position` always belongs to whoever is about to move. The mover's own board after a move is `position ^ mask`, which is the identity the win check uses.

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt

H, W = 6, 7  # rows, columns
H1 = H + 1  # one sentinel row per column, so shifts cannot wrap between columns
H2 = H + 2
TOP_PLAYABLE = [1 << (c * H1 + H - 1) for c in range(W)]


def new_game():
    """(position, mask): the mover's discs, and every disc on the board."""
    return 0, 0


def legal_moves(mask):
    return [c for c in range(W) if not mask & TOP_PLAYABLE[c]]


def play(position, mask, col):
    """Drop a disc, then hand the board to the other player.

    `position` always belongs to whoever is about to move, so the swap happens
    first and the new disc lands in `mask` afterwards. The mover's own board
    after the move is `position ^ mask`.
    """
    position ^= mask
    mask |= mask + (1 << (col * H1))
    return position, mask


def is_win(pos):
    """Four in a row anywhere in `pos`, in four shifts.

    Shifting by 1 walks a column, by H1 a row, and by H and H2 the two
    diagonals. `pos & (pos >> k)` marks the start of every pair; ANDing that
    with itself shifted by 2k marks the start of every four.
    """
    for k in (1, H, H1, H2):
        m = pos & (pos >> k)
        if m & (m >> (2 * k)):
            return True
    return False


def show(position, mask):
    theirs, ours = position, position ^ mask
    rows = []
    for r in range(H - 1, -1, -1):
        row = []
        for c in range(W):
            bit = 1 << (c * H1 + r)
            row.append("X" if ours & bit else "O" if theirs & bit else ".")
        rows.append(" ".join(row))
    return "\n".join(rows)


p, m = new_game()
for col in (3, 3, 4, 2, 4):
    p, m = play(p, m, col)
print(show(p, m))
print("legal moves:", legal_moves(m))

## 2. The part that sounds like it cannot work

A rollout plays uniformly at random until somebody wins. No heuristic, no policy, no learning. The claim the whole method rests on is that the average of enough of these ranks moves correctly, even though no individual playout resembles real play.

Note the second number the cell prints: from an empty board, random play favours the first player only slightly. Connect Four is a first-player win under perfect play, which is why every match below swaps sides.

In [ ]:
def random_rollout(position, mask, rng):
    """Play uniformly at random to the end. +1, 0 or -1 for the player to move.

    No heuristic, no policy, no learning. The claim the whole method rests on
    is that the average of enough of these ranks moves correctly.
    """
    to_move = 1
    while True:
        moves = legal_moves(mask)
        if not moves:
            return 0
        position, mask = play(position, mask, moves[rng.randrange(len(moves))])
        if is_win(position ^ mask):
            return to_move
        to_move = -to_move


rng = random.Random(0)
t0 = time.perf_counter()
N = 5000
results = [random_rollout(0, 0, rng) for _ in range(N)]
dt = (time.perf_counter() - t0) / N
print(f"{1 / dt:,.0f} rollouts per second, {dt * 1e6:.0f} us each")
print("from an empty board, first player wins", f"{results.count(1) / N:.1%} of random games")

## 3. Four phases and one formula

**Selection** walks down from the root taking the highest UCT score. **Expansion** adds exactly one child, which is why the tree ends up with exactly `budget + 1` nodes. **Simulation** plays out at random. **Backpropagation** carries the result up, flipping sign at each step because the mover alternates.

UCT is UCB1 applied to a tree: mean result plus `c * sqrt(log(parent visits) / visits)`. The first term prefers what has been winning, the second prefers what has been tried least.

The `q` property is where this goes wrong silently. Read its docstring before changing anything.

In [ ]:
class Node:
    __slots__ = ("position", "mask", "parent", "move", "children", "untried", "visits", "value")

    def __init__(self, position, mask, parent=None, move=None):
        self.position, self.mask = position, mask
        self.parent, self.move = parent, move
        self.children, self.untried = [], legal_moves(mask)
        self.visits, self.value = 0, 0.0

    @property
    def q(self):
        """Mean result of this move, from the point of view of whoever picks it.

        `value` is stored from the point of view of the player to move *at* this
        node, which is the opponent of the player choosing the move that leads
        here, so reading it from the chooser's side means negating. Get this
        sign wrong and the search still runs, still builds a sensible looking
        tree, and loses every game to a uniformly random opponent.
        """
        return -self.value / self.visits if self.visits else 0.0

    def uct(self, c):
        return self.q + c * math.sqrt(math.log(self.parent.visits) / self.visits)


def backup(node, result):
    """Walk to the root, negating at every step because the mover alternates."""
    while node is not None:
        node.visits += 1
        node.value += result
        result, node = -result, node.parent


def mcts_move(position, mask, budget, c=1.4, rng=None, return_root=False):
    rng = rng or random.Random(0)
    root = Node(position, mask)
    for _ in range(budget):
        node = root
        while not node.untried and node.children:                    # 1. selection
            node = max(node.children, key=lambda n: n.uct(c))
        if node.untried:                                             # 2. expansion
            col = node.untried.pop(rng.randrange(len(node.untried)))
            child = Node(*play(node.position, node.mask, col), parent=node, move=col)
            node.children.append(child)
            node = child
            if is_win(node.position ^ node.mask):
                backup(node, -1.0)                # the move just played wins
                continue
        backup(node, float(random_rollout(node.position, node.mask, rng)))  # 3, 4
    best = max(root.children, key=lambda n: n.visits)
    return (best.move, root) if return_root else best.move


col, root = mcts_move(p, m, 1000, rng=random.Random(0), return_root=True)
print("chosen column:", col)
for ch in sorted(root.children, key=lambda n: n.move):
    print(f"  col {ch.move}: {ch.visits:4d} visits, mean {ch.q:+.3f}")

## 4. The control arm

Depth-limited alpha-beta needs an evaluation function for the positions it stops at, and writing one is the domain-specific work MCTS avoids entirely. This one counts open four-in-a-row lines by how many cells each side owns.

That asymmetry is the comparison: one player is told which patterns are good, and the other is told nothing.

In [ ]:
def build_line_masks():
    """Every four-in-a-row as one 49-bit mask. 69 of them on a 7x6 board."""
    masks = []
    for r in range(H):
        for c in range(W):
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                cells = [(r + dr * i, c + dc * i) for i in range(4)]
                if all(0 <= rr < H and 0 <= cc < W for rr, cc in cells):
                    mm = 0
                    for rr, cc in cells:
                        mm |= 1 << (cc * H1 + rr)
                    masks.append(mm)
    return masks


LINE_MASKS = build_line_masks()
SCORES = (0, 1, 4, 32, 1000)


def evaluate(position, mask):
    """A hand-written score for a non-terminal position, from the mover's side.

    Alpha-beta cannot search Connect Four to the end in any budget used here,
    so it needs this. MCTS needs no equivalent, and that asymmetry is the
    comparison this notebook is making.
    """
    ours, theirs = position ^ mask, position
    score = 0
    for line in LINE_MASKS:
        o, t = (ours & line).bit_count(), (theirs & line).bit_count()
        if o and t:
            continue                       # blocked, worth nothing to either side
        score += SCORES[o] - SCORES[t]
    return score


def alphabeta_move(position, mask, depth, rng=None):
    rng = rng or random.Random(0)
    best, best_score = None, -math.inf
    moves = legal_moves(mask)
    rng.shuffle(moves)
    for col in moves:
        pp, mm = play(position, mask, col)
        if is_win(pp ^ mm):
            return col
        score = -_ab(pp, mm, depth - 1, -math.inf, math.inf)
        if score > best_score:
            best, best_score = col, score
    return best


def _ab(position, mask, depth, alpha, beta):
    moves = legal_moves(mask)
    if not moves:
        return 0
    if depth == 0:
        return evaluate(position, mask)
    best = -math.inf
    for col in moves:
        pp, mm = play(position, mask, col)
        score = 100000 + depth if is_win(pp ^ mm) else -_ab(pp, mm, depth - 1, -beta, -alpha)
        best = max(best, score)
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    return best


print("alpha-beta depth 4 picks column", alphabeta_move(p, m, 4, rng=random.Random(0)))

## 5. Does it beat a random player

The first thing to build, and the check that catches a broken search. An agent that cannot beat random is not weak, it is wrong. Twenty-five rollouts per move should already win almost every game.

In [ ]:
def play_match(agent_a, agent_b, seed=0, a_first=True):
    """+1 if agent_a wins, -1 if agent_b wins, 0 for a draw."""
    rng = random.Random(seed)
    position, mask = new_game()
    turn = "a" if a_first else "b"
    agents = {"a": agent_a, "b": agent_b}
    while True:
        if not legal_moves(mask):
            return 0
        position, mask = play(position, mask, agents[turn](position, mask, rng))
        if is_win(position ^ mask):
            return 1 if turn == "a" else -1
        turn = "b" if turn == "a" else "a"


def mcts_agent(budget, c=1.4):
    return lambda pos, msk, rng: mcts_move(pos, msk, budget, c=c, rng=rng)


def alphabeta_agent(depth):
    return lambda pos, msk, rng: alphabeta_move(pos, msk, depth, rng=rng)


def random_agent():
    return lambda pos, msk, rng: legal_moves(msk)[rng.randrange(len(legal_moves(msk)))]


def duel(a, b, games=20, seed0=0):
    wins = draws = 0
    for i in range(games):
        r = play_match(a, b, seed=seed0 + i, a_first=(i % 2 == 0))
        wins += r == 1
        draws += r == 0
    return wins / games, draws / games


for budget in (10, 25, 50, 100):
    w, d = duel(mcts_agent(budget), random_agent(), 20)
    print(f"budget {budget:4d} vs random:      win {w:.2f}  draw {d:.2f}")

## 6. The headline, node-matched

Budget 75 costs about 11,000 nodes per game and alpha-beta depth 4 about 10,000, so this is close to a fair comparison of work rather than of parameters. Expect roughly 0.7 over enough games; twenty games has a standard error of about 10 points, so do not read too much into a single run.

In [ ]:
t0 = time.perf_counter()
w, d = duel(mcts_agent(75), alphabeta_agent(4), 20)
print(f"MCTS budget 75 vs alpha-beta depth 4:  win {w:.2f}  draw {d:.2f}")
print(f"({time.perf_counter() - t0:.0f}s; roughly node-matched at ~11k vs ~10k per game)")

## 7. The exploration constant

Setting `c = 0` removes the exploration term and turns selection into greed on the current average. Everything from about 0.5 to 2.0 behaves the same. The commonly quoted `sqrt(2)` sits in that range, and so does 0.5.

In [ ]:
sweep = {}
for c in (0.0, 0.25, 0.5, 1.0, 1.4, 3.0):
    sweep[c], _ = duel(mcts_agent(200, c=c), alphabeta_agent(4), 16)
    print(f"c = {c:<4}  win {sweep[c]:.2f}")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(list(sweep), list(sweep.values()), "o-", color="#1f9e9e", lw=2)
ax.set_xlabel("UCT exploration constant c")
ax.set_ylabel("win rate vs alpha-beta depth 4")
ax.grid(alpha=0.25)
plt.show()

## 8. An exact oracle, and a seven-ply trap

A shallow exact solver is not a heuristic: it searches to terminal positions or to the horizon and returns 0 for anything unresolved, so a +1 is a proof.

The position below has exactly one move that wins by force, and the win does not appear until seven plies deep.

In [ ]:
def solve(position, mask, plies, alpha=-1, beta=1):
    """Exact result within `plies`, from the mover's side: +1, 0 or -1.

    Not a heuristic. It searches to terminal positions or to the horizon and
    returns 0 for anything it cannot resolve, so a +1 is a proof of a forced
    win. That is what makes it usable as ground truth.
    """
    moves = legal_moves(mask)
    if not moves or plies == 0:
        return 0
    best = -1
    for col in moves:
        pp, mm = play(position, mask, col)
        if is_win(pp ^ mm):
            return 1
        v = -solve(pp, mm, plies - 1, -beta, -alpha)
        best = max(best, v)
        alpha = max(alpha, best)
        if alpha >= beta:
            break
    return best


# A position reached by 14 legal moves. Column 3 wins by force and nothing
# else does, and the win does not appear until seven plies deep, so it cannot
# be found by looking one move ahead.
tp, tm = new_game()
for col in (1, 3, 1, 2, 3, 1, 6, 2, 5, 4, 6, 4, 3, 6):
    tp, tm = play(tp, tm, col)
print(show(tp, tm))
print("exact value of each move within 7 plies:")
for col in legal_moves(tm):
    pp, mm = play(tp, tm, col)
    print(f"  col {col}: {1 if is_win(pp ^ mm) else -solve(pp, mm, 6):+d}")

## 9. Anytime

The answer is the most-visited child of the root, which is defined from the first rollout onwards. Stop the search whenever you like and you have its best move so far. Watch how few rollouts it takes to lock onto the winning column.

In [ ]:
for budget in (10, 25, 50, 100, 1000):
    picks = [mcts_move(tp, tm, budget, rng=random.Random(s)) for s in range(10)]
    print(f"budget {budget:5d}: {picks.count(3)}/10 runs return the winning column")

## Exercises

1. **Break the sign.** Change `q` to return `self.value / self.visits` and re-run the match against a random opponent. The tree still grows correctly and the agent loses every game. This is the failure mode worth having seen once.
2. **Find the crossover.** Sweep the budget against alpha-beta depth 3, counting nodes on both sides. Below roughly 2,000 nodes the shallow exact search wins. Where exactly does your curve cross?
3. **A better rollout.** Replace the uniform random playout with one that takes an immediate win and blocks an immediate loss when either exists. Measure the win rate at a fixed budget before and after. This is the cheapest step on the road to AlphaZero's value network.
4. **Most visits or best mean.** Return `max(root.children, key=lambda n: n.q)` instead of the most-visited child and measure the difference. Then work out why visit count is the more reliable statistic.
5. **Tree reuse.** After making a move, keep the subtree under the chosen child instead of discarding it, and start the next search from there. How much budget does that recover?


## Further reading

- Coulom (2006), [Efficient Selectivity and Backup Operators in Monte-Carlo Tree Search](https://inria.hal.science/inria-00116992/document)
- Kocsis & Szepesvari (2006), [Bandit based Monte-Carlo Planning](https://link.springer.com/chapter/10.1007/11871842_29), the UCT paper
- Auer, Cesa-Bianchi & Fischer (2002), [Finite-time Analysis of the Multiarmed Bandit Problem](https://link.springer.com/article/10.1023/A:1013689704352), UCB1
- Browne et al. (2012), [A Survey of Monte Carlo Tree Search Methods](https://ieeexplore.ieee.org/document/6145622)
- Silver et al. (2017), [Mastering Chess and Shogi by Self-Play](https://arxiv.org/abs/1712.01815), AlphaZero
- [Q-Learning for Games: Teaching an Agent Tic-Tac-Toe Through Self-Play](https://sesen.ai/blog/q-learning-games-tic-tac-toe-self-play), the post this one is the sequel to
